In [1]:
import numpy as np

# 1. IMPORTAÇÕES DA BASE
from my_lr import (
    funcao_discriminante_linear,
    funcao_sigmoide,
    atualizar_parametros
)

# =========================================================================
# 2. COMPONENTES MATEMÁTICOS HÍBRIDOS (Focal Loss + AHIW)
# =========================================================================

def calcular_focal_loss_ahiw(y_real, y_probabilidade, pesos_historicos, alpha=0.25, gamma_focal=2.0):
    """
    Calcula a Focal Loss e escala o custo individual pelo histórico da amostra (AHIW).
    """
    eps = 1e-15
    p = np.clip(y_probabilidade, eps, 1 - eps)
    
    p_t = np.where(y_real == 1, p, 1 - p)
    alpha_t = np.where(y_real == 1, alpha, 1 - alpha)
    
    # 1. Custo Focal Individual
    fator_modulador = np.power((1 - p_t), gamma_focal)
    custo_focal_individual = -alpha_t * fator_modulador * np.log(p_t)
    
    # 2. Multiplicação pelo Histórico (AHIW)
    custo_ponderado = custo_focal_individual * pesos_historicos
    
    return np.mean(custo_ponderado)

def calcular_gradientes_focal_loss_ahiw(X, y_real, y_probabilidade, pesos_historicos, alpha=0.25, gamma_focal=2.0):
    """
    Calcula os gradientes da Focal Loss e aplica a alavancagem escalar do AHIW.
    """
    N = len(y_real)
    eps = 1e-15
    p = np.clip(y_probabilidade, eps, 1 - eps)
    
    # 1. Derivada da Loss em relação a p (Focal Loss)
    dl_dp_1 = alpha * (gamma_focal * (1 - p)**(gamma_focal - 1) * np.log(p) - (1 - p)**gamma_focal / p)
    dl_dp_0 = -(1 - alpha) * (gamma_focal * p**(gamma_focal - 1) * np.log(1 - p) - p**gamma_focal / (1 - p))
    dl_dp = np.where(y_real == 1, dl_dp_1, dl_dp_0)
    
    # 2. Derivada da Sigmóide
    dp_dz = p * (1 - p)
    
    # 3. Resíduo Focal
    erro_residual_focal = dl_dp * dp_dz
    
    # 4. Aplicação do AHIW no vetor de gradientes
    erro_residual_final = erro_residual_focal * pesos_historicos
    
    gradiente_pesos = np.dot(X.T, erro_residual_final) / N
    gradiente_bias = np.sum(erro_residual_final) / N
    
    return gradiente_pesos, gradiente_bias

# =========================================================================
# 3. ROTINA DE TREINAMENTO HÍBRIDA
# =========================================================================

def treinar_lr_hibrida(X, y, alpha_lr=0.01, epocas=1000, 
                       alpha_focal=0.25, gamma_focal=2.0, gama_historico=1.2):
    """
    Loop de treino que gere a atualização dos pesos da regressão e 
    do vetor de estado (AHIW) em simultâneo.
    """
    N_amostras, N_caracteristicas = X.shape
    
    pesos = np.zeros(N_caracteristicas)
    bias = 0.0
    
    # Inicialização do AHIW: Todas as amostras começam com peso 1
    pesos_historicos = np.ones(N_amostras)
    
    historico_custo = []
    
    for epoca in range(epocas):
        # --- 1. Forward Pass ---
        z = funcao_discriminante_linear(X, pesos, bias)
        y_prob = funcao_sigmoide(z)
        
        # --- 2. Custo ---
        custo = calcular_focal_loss_ahiw(
            y, y_prob, pesos_historicos, alpha=alpha_focal, gamma_focal=gamma_focal
        )
        historico_custo.append(custo)
        
        # --- 3. Backward Pass ---
        grad_pesos, grad_bias = calcular_gradientes_focal_loss_ahiw(
            X, y, y_prob, pesos_historicos, alpha=alpha_focal, gamma_focal=gamma_focal
        )
        
        # --- 4. Atualização dos Parâmetros da Rede ---
        pesos, bias = atualizar_parametros(pesos, bias, grad_pesos, grad_bias, alpha_lr)
        
        # --- 5. Atualização do Vetor de Memória AHIW ---
        # Mede o erro absoluto da iteração atual
        erro_absoluto = np.abs(y_prob - y)
        
        # Atualização exponencial: amostras com maior erro ganham mais peso
        pesos_historicos = pesos_historicos * np.exp(gama_historico * erro_absoluto)
        
        # Normalização rigorosa para manter a escala dos gradientes sob controlo
        pesos_historicos = (pesos_historicos / np.sum(pesos_historicos)) * N_amostras
        
    return pesos, bias, historico_custo

Com as modificações devidamente implementadas, é possível utilizar do novo algoritmo para realizar as etapas de teste/treino conforme logistic regression tradicional e feita anteriormente nese Assigment.

In [ ]:
# Hiperparâmetros
ALPHA_LR        = 0.01    # taxa de aprendizagem
EPOCAS          = 1000
ALPHA_FOCAL     = 0.75    # peso da classe minoritária (ajusta conforme proporção)
GAMMA_FOCAL     = 2.0     # supressão de exemplos fáceis
GAMA_HISTORICO  = 1.2     # agressividade do AHIW

pesos, bias, historico_custo = treinar_lr_hibrida(
    X_train, y_train,
    alpha_lr       = ALPHA_LR,
    epocas         = EPOCAS,
    alpha_focal    = ALPHA_FOCAL,
    gamma_focal    = GAMMA_FOCAL,
    gama_historico = GAMA_HISTORICO
)